# Behind the Feed — Phase 2
## Baseline News Recommendation System

Team: Data Bees

### Objective
Build a basic content-based recommendation system using
our cleaned GDELT dataset.

Users will provide their topic preferences, and the system
will recommend articles based on those preferences.

This will serve as the baseline for comparing our
diversity-aware recommendation system.

In [2]:
import pandas as pd
import numpy as np



In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Get the dataset path
data_path = Path("../data/gdelt_clean.csv")

# Load dataset
df = pd.read_csv(data_path, keep_default_na=False)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (15395, 14)


,article_id,title,publication_date,country,country_code,continent,topic,publisher,article_url,all_countries,all_country_codes,all_continents,primary_topic,original_themes
0,20260714120000-0,Man accused of assigning attacks on venues fac...,2026-07-14,Australia,AU,Oceania,Conflict and Security; Business and Economy; H...,irrigator.com.au,https://www.irrigator.com.au/story/9310322/man...,Australia,AU,Oceania,Conflict and Security,TAX_FNCACT;TAX_FNCACT_MAN;ARREST;SOC_GENERALCR...
1,20260714120000-1,"US strikes Iran, vows to reimpose naval blockade",2026-07-14,"Iran, Islamic Republic of",IR,Asia,Business and Economy; Conflict and Security; E...,vanguardngr.com,https://www.vanguardngr.com/2026/07/us-strikes...,"Iran, Islamic Republic of; United Kingdom; Uni...",IR; GB; US; PK; JO; QA; BH; OM; AE,Asia; Europe; North America,Business and Economy,LEADER;TAX_FNCACT;TAX_FNCACT_PRESIDENT;USPEC_P...
2,20260714120000-2,'ISIS bride' allegedly urged kids to be like t...,2026-07-14,Australia,AU,Oceania,Conflict and Security; Health; Society and Cul...,queanbeyanage.com.au,https://www.queanbeyanage.com.au/story/9310787...,Australia; Syrian Arab Republic,AU; SY,Oceania; Asia,Conflict and Security,TAX_FNCACT;TAX_FNCACT_CHILDREN;TAX_TERROR_GROU...
3,20260714120000-3,RNLI Barmouth called out several times on busy...,2026-07-14,United Kingdom,GB,Europe,Conflict and Security; Environment and Climate,countytimes.co.uk,https://www.countytimes.co.uk/news/26278293.rn...,United Kingdom,GB,Europe,Conflict and Security,KILL;CRISISLEX_CRISISLEXREC;CRISISLEX_T02_INJU...
4,20260714120000-5,Lionel Messi's lucky omen? England's World Cup...,2026-07-14,Netherlands,NL,Europe,Unknown,lbc.co.uk,https://www.lbc.co.uk/article/lionel-messi-eng...,Netherlands; Portugal; Brazil; United States; ...,NL; PT; BR; US; IT; NO; UY; AR; ES; FR; CH; QA,Europe; South America; North America; Asia,Unknown,


In [4]:
# Convert semicolon-separated topics into individual topic labels
df["topic_list"] = df["topic"].apply(
    lambda x: [t.strip() for t in x.split(";") if t.strip()]
)

# Find all unique topic categories
all_topics = sorted({
    topic
    for topics in df["topic_list"]
    for topic in topics
})

print("Total unique topics:", len(all_topics))

print("\nAvailable topics:")
for topic in all_topics:
    print("-", topic)

Total unique topics: 11

Available topics:
- Business and Economy
- Conflict and Security
- Education
- Environment and Climate
- Health
- Other
- Politics and Government
- Science
- Society and Culture
- Technology
- Unknown


In [5]:
from collections import Counter

topic_counts = Counter(
    topic
    for topics in df["topic_list"]
    for topic in topics
)

topic_distribution = (
    pd.DataFrame(
        topic_counts.items(),
        columns=["Topic", "Article Count"]
    )
    .sort_values("Article Count", ascending=False)
)

topic_distribution

,Topic,Article Count
0,Conflict and Security,8824
1,Business and Economy,8686
3,Politics and Government,8233
6,Environment and Climate,7414
4,Society and Culture,5999
2,Health,5345
5,Technology,3933
8,Education,3795
9,Science,1437
7,Unknown,1073


In [6]:
user_profile = {
    "Technology": 0.8,
    "Business and Economy": 0.6,
    "Health": 0.2
}

user_profile

{'Technology': 0.8, 'Business and Economy': 0.6, 'Health': 0.2}

In [7]:
# Topics available for user preferences
selectable_topics = set(all_topics) - {"Unknown", "Other"}

def calculate_relevance(topic_list, user_profile):

    valid_topics = [
        topic for topic in topic_list
        if topic in selectable_topics
    ]

    if not valid_topics:
        return 0.0

    scores = [
        user_profile.get(topic, 0.0)
        for topic in valid_topics
    ]

    return sum(scores) / len(scores)


df["relevance_score"] = df["topic_list"].apply(
    lambda topics: calculate_relevance(topics, user_profile)
)

In [8]:
baseline_feed = (
    df[df["relevance_score"] > 0]
    .sort_values(
        "relevance_score",
        ascending=False
    )
    .head(10)
)

baseline_feed[
    [
        "title",
        "topic",
        "country",
        "publisher",
        "relevance_score"
    ]
]

,title,topic,country,publisher,relevance_score
4625,Uldale road set to close for five days this week,Technology,United Kingdom,newsandstar.co.uk,0.8
10581,Markets brace for higher rates as AI safety de...,Technology,United States,actuarialpost.co.uk,0.8
9754,From Rio to Future Past: Duran Duran Celebrate...,Technology,United Kingdom,949kcmo.com,0.8
4224,Why the data shows Guimarães deal could be ano...,Technology,Italy,aol.co.uk,0.8
13817,Who is Rohit Prasad? Former Amazon AI executiv...,Technology,United States,livemint.com,0.8
4417,Quint Digital Limited Q1 FY 2026-27: Building ...,Technology,Nepal,nepalnational.com,0.8
6691,Chinese Humanoid Robot Shatters Usain Bolt's L...,Technology,China,lankanewspapers.com,0.8
13143,Sherwin-Williams Opens Expanded Statesville Plant,Technology,United States,whky.com,0.8
7210,Hugging Face Unveils $400 Duck-Like Robot That...,Technology,China,deccanchronicle.com,0.8
6730,Data centre proposed for Charlton industrial s...,Technology,United Kingdom,yahoo.com,0.8


In [9]:
baseline_feed = (
    df[df["relevance_score"] > 0]
    .sort_values(
        by=["relevance_score", "article_id"],
        ascending=[False, True]
    )
    .head(10)
)

baseline_feed[
    [
        "article_id",
        "title",
        "topic",
        "country",
        "publisher",
        "relevance_score"
    ]
]

,article_id,title,topic,country,publisher,relevance_score
374,20260714120000-455,Consumer Trends Reveal Growing Demand For Inte...,Technology,United States,explosion.com,0.8
2685,20260729120000-497,"As search and social traffic dries up, local n...",Technology,United States,poynter.org,0.8
2918,20260729120000-780,Creative Realities Named Official Digital Sign...,Technology,Philippines,manilatimes.net,0.8
3608,20260803120000-357,Deloitte inks 5-year tie-up with The Kallang t...,Technology,Philippines,businesstimes.com.sg,0.8
3630,20260803120000-391,Foxconn vendor Agile Robots acquires majority ...,Technology,India,thehindubusinessline.com,0.8
4417,20260808120000-315,Quint Digital Limited Q1 FY 2026-27: Building ...,Technology,Nepal,nepalnational.com,0.8
4625,20260808120000-581,Uldale road set to close for five days this week,Technology,United Kingdom,newsandstar.co.uk,0.8
4224,20260808120000-85,Why the data shows Guimarães deal could be ano...,Technology,Italy,aol.co.uk,0.8
5497,20260813120000-1009,Age-restricted: Santa Maria follows the county...,Technology,United States,santamariasun.com,0.8
5527,20260813120000-1049,Bickington road closures planned for major gas...,Technology,United Kingdom,midweekherald.co.uk,0.8


In [10]:
print("Baseline Evaluation (Top 10)")
print("----------------------------")

print("Unique countries:",
      baseline_feed["country"].nunique())

print("Unique publishers:",
      baseline_feed["publisher"].nunique())

print("Unique primary topics:",
      baseline_feed["primary_topic"].nunique())

print("Average relevance:",
      round(baseline_feed["relevance_score"].mean(), 3))

Baseline Evaluation (Top 10)
----------------------------
Unique countries: 6
Unique publishers: 10
Unique primary topics: 1
Average relevance: 0.8


In [11]:
def recommend_baseline(df, user_profile, top_k=10):

    recommendations = df.copy()

    # Calculate relevance based on user interests
    recommendations["relevance_score"] = (
        recommendations["topic_list"].apply(
            lambda topics: calculate_relevance(
                topics,
                user_profile
            )
        )
    )

    # Keep articles with positive relevance
    recommendations = recommendations[
        recommendations["relevance_score"] > 0
    ]

    # Rank articles by relevance
    recommendations = recommendations.sort_values(
        by=["relevance_score", "article_id"],
        ascending=[False, True]
    )

    return recommendations.head(top_k).copy()

In [12]:
user_2 = {
    "Environment and Climate": 0.9,
    "Health": 0.7,
    "Science": 0.5
}

feed_2 = recommend_baseline(
    df,
    user_2,
    top_k=10
)

feed_2[
    [
        "title",
        "topic",
        "country",
        "publisher",
        "relevance_score"
    ]
]

,title,topic,country,publisher,relevance_score
841,Poll: Majority of UK public want to see equal ...,Environment and Climate,United Kingdom,businessgreen.com,0.9
109,"Chicago, IL weather: Hot across area Tuesday w...",Environment and Climate,United States,abc7chicago.com,0.9
118,Heat Advisory in Effect Noon Today – 8 PM Wedn...,Environment and Climate,United States,whbl.com,0.9
145,Offaly Fire and Rescue Service urges caution a...,Environment and Climate,Ireland,offalyindependent.ie,0.9
204,Old Farmer's Almanac Fall Forecast for New York,Environment and Climate,United States,961theeagle.com,0.9
31,2027 Triumph Speed Twin 1200 TFC revealed,Environment and Climate,Australia,mcnews.com.au,0.9
405,Pembrokeshire dogs on beaches questions to be ...,Environment and Climate,United Kingdom,milfordmercury.co.uk,0.9
515,Rainy Pattern Continues Today Before Summer He...,Environment and Climate,United States,chattooga1180.com,0.9
565,Israeli colonists destroy dozens of olive tree...,Environment and Climate,Israel,english.wafa.ps,0.9
605,New Forest residents warned of bin collection ...,Environment and Climate,United Kingdom,dailyecho.co.uk,0.9


In [13]:
user_profiles = {
    "Tech Enthusiast": {
        "Technology": 0.8,
        "Business and Economy": 0.6,
        "Health": 0.2
    },

    "Environment Explorer": {
        "Environment and Climate": 0.9,
        "Science": 0.7,
        "Technology": 0.3
    },

    "Health Researcher": {
        "Health": 0.9,
        "Science": 0.8,
        "Education": 0.4
    }
}

In [14]:
for user_name, profile in user_profiles.items():

    feed = recommend_baseline(
        df,
        profile,
        top_k=10
    )

    print(f"\n{user_name}")
    print("-" * 40)

    print("Countries:", feed["country"].nunique())
    print("Publishers:", feed["publisher"].nunique())
    print("Primary topics:", feed["primary_topic"].nunique())
    print("Average relevance:",
          round(feed["relevance_score"].mean(), 3))


Tech Enthusiast
----------------------------------------
Countries: 6
Publishers: 10
Primary topics: 1
Average relevance: 0.8

Environment Explorer
----------------------------------------
Countries: 5
Publishers: 10
Primary topics: 1
Average relevance: 0.9

Health Researcher
----------------------------------------
Countries: 5
Publishers: 8
Primary topics: 1
Average relevance: 0.9


In [15]:
baseline_results = []

for user_name, profile in user_profiles.items():

    feed = recommend_baseline(
        df,
        profile,
        top_k=10
    )

    baseline_results.append({
        "user_profile": user_name,
        "unique_countries": feed["country"].nunique(),
        "unique_publishers": feed["publisher"].nunique(),
        "unique_primary_topics": feed["primary_topic"].nunique(),
        "average_relevance": round(
            feed["relevance_score"].mean(), 3
        )
    })

baseline_evaluation = pd.DataFrame(baseline_results)

baseline_evaluation

,user_profile,unique_countries,unique_publishers,unique_primary_topics,average_relevance
0,Tech Enthusiast,6,10,1,0.8
1,Environment Explorer,5,10,1,0.9
2,Health Researcher,5,8,1,0.9


In [16]:
baseline_evaluation.to_csv(
    "baseline_evaluation.csv",
    index=False
)

## Full Dataset for Recommendation

In [17]:
# Keep all articles
all_candidates = df.copy()

# Calculate relevance for the Tech Enthusiast profile
all_candidates["relevance_score"] = (
    all_candidates["topic_list"].apply(
        lambda topics: calculate_relevance(
            topics,
            user_profiles["Tech Enthusiast"]
        )
    )
)

print("Total articles:", len(all_candidates))

print("\nRelevance score distribution:")
print(all_candidates["relevance_score"].value_counts().sort_index())

Total articles: 15395

Relevance score distribution:
relevance_score
0.000000    4270
0.028571       1
0.033333      48
0.040000     186
0.050000     314
0.066667     425
0.085714       9
0.100000     119
0.100000     345
0.114286     252
0.120000     556
0.125000       3
0.133333     551
0.142857      48
0.150000     841
0.160000     725
0.166667     106
0.175000      46
0.177778      17
0.177778      78
0.200000    1046
0.200000    1189
0.228571      55
0.228571     409
0.233333     464
0.250000      58
0.266667     705
0.280000     422
0.300000     679
0.320000      37
0.320000     120
0.333333      25
0.350000     284
0.400000       5
0.400000     246
0.466667     156
0.500000       5
0.533333      10
0.600000     444
0.700000      46
0.800000      50
Name: count, dtype: int64


In [18]:
print("Total articles:", len(all_candidates))

print("Unique article IDs:",
      all_candidates["article_id"].nunique())

print("Missing relevance scores:",
      all_candidates["relevance_score"].isna().sum())

print("Minimum relevance:",
      all_candidates["relevance_score"].min())

print("Maximum relevance:",
      all_candidates["relevance_score"].max())

print("Articles matching user interests:",
      (all_candidates["relevance_score"] > 0).sum())

Total articles: 15395
Unique article IDs: 15395
Missing relevance scores: 0
Minimum relevance: 0.0
Maximum relevance: 0.8
Articles matching user interests: 11125


## Reusable Relevance Scoring Function

In [19]:
def score_all_articles(df, user_profile):
    """
    Calculate relevance scores for all articles
    based on a user's weighted topic preferences.
    """

    scored_df = df.copy()

    scored_df["relevance_score"] = (
        scored_df["topic_list"].apply(
            lambda topics: calculate_relevance(
                topics,
                user_profile
            )
        )
    )

    return scored_df

In [20]:
scored_articles = score_all_articles(
    df,
    user_profiles["Tech Enthusiast"]
)

print("Total scored articles:", len(scored_articles))

scored_articles[
    [
        "article_id",
        "title",
        "topic",
        "country",
        "publisher",
        "relevance_score"
    ]
].head()

Total scored articles: 15395


,article_id,title,topic,country,publisher,relevance_score
0,20260714120000-0,Man accused of assigning attacks on venues fac...,Conflict and Security; Business and Economy; H...,Australia,irrigator.com.au,0.266667
1,20260714120000-1,"US strikes Iran, vows to reimpose naval blockade",Business and Economy; Conflict and Security; E...,"Iran, Islamic Republic of",vanguardngr.com,0.233333
2,20260714120000-2,'ISIS bride' allegedly urged kids to be like t...,Conflict and Security; Health; Society and Cul...,Australia,queanbeyanage.com.au,0.066667
3,20260714120000-3,RNLI Barmouth called out several times on busy...,Conflict and Security; Environment and Climate,United Kingdom,countytimes.co.uk,0.000000
4,20260714120000-5,Lionel Messi's lucky omen? England's World Cup...,Unknown,Netherlands,lbc.co.uk,0.000000


In [21]:
df.loc[
    df["title"].str.contains(
        "Uldale road set to close",
        case=False,
        na=False
    ),
    ["title", "topic", "primary_topic", "original_themes"]
]

,title,topic,primary_topic,original_themes
4625,Uldale road set to close for five days this week,Technology,Technology,CLOSURE;AFFECT;TRAFFIC;DISABILITY;WB_168_ROADS...


In [22]:
article = df.loc[
    df["title"].str.contains(
        "Uldale road set to close",
        case=False,
        na=False
    )
]

print(article["original_themes"].iloc[0])

CLOSURE;AFFECT;TRAFFIC;DISABILITY;WB_168_ROADS_AND_HIGHWAYS;WB_135_TRANSPORT;WB_1809_HIGHWAYS;WB_1803_TRANSPORT_INFRASTRUCTURE;CRISISLEX_C08_TELECOM;


# Baseline System Summary

## Objective

Build a simple topic-based news recommendation system using our cleaned GDELT dataset.

This system will serve as the baseline for comparing our improved diversity-aware recommendation system.

## Dataset

- 15,395 unique articles.
- 14 original columns.
- Multiple topic labels per article.
- Country and publisher information.
- Article titles and URLs.

## User Preferences

The baseline uses topic preference weights between 0 and 1.

Example:

- Technology: 0.8
- Business and Economy: 0.6
- Health: 0.2

We have created three example user profiles to test the baseline.

**For the final website:** Users will simply select their preferred topics. Each selected topic will initially receive a weight of 1.0.

## Recommendation Method

1. Read each article's topic labels.
2. Match the topics with the user's preferences.
3. Calculate the average preference weight across valid topics.
4. Rank articles by relevance score.
5. Return the top 10 articles.

The baseline focuses only on topic relevance. It does not explicitly optimize diversity.

## Reusable Functions

**`calculate_relevance()`**

Calculates the relevance score of one article based on the user's topic preferences.

**`score_all_articles()`**

Calculates relevance scores for all 15,395 articles without removing articles with zero relevance.

**`recommend_baseline()`**

Ranks articles by relevance and returns the top recommendations with positive relevance scores.

## Important: Using the Full Dataset

For the improved recommendation system, we will work with all 15,395 articles rather than only the baseline's top recommendations.

This is important because articles outside the user's existing interests may still be useful for discovery.

For example, a Technology user might discover relevant articles about Healthcare or Climate Change.

Shahida and Suraiya should use the complete article dataset when developing their components.

## Initial Findings

- Baseline recommendations mainly focus on users' strongest interests.
- The baseline already shows some geographic and publisher variety.
- Topic diversity is limited in the initial top-10 results when measured using `primary_topic`.
- Some topic labels appear noisy.
- The baseline does not explicitly optimize diversity.

These findings give us a starting point for evaluating the improved system.

## Next Development

### 1. Interest Bridging — Shahida

Develop an NLP-based method using TF-IDF and cosine similarity.

Goal: Identify articles that connect users' existing interests with new but related topics.

Example: Technology → AI in Healthcare.

### 2. Geographic and Publisher Diversity — Suraiya

Develop a diversity-aware re-ranking method.

Goal: Recommend relevant articles associated with different countries and publishers.

The method should balance relevance and diversity.

### 3. Integration

Combine:

- Baseline relevance scores.
- Interest Bridging.
- Geographic diversity.
- Publisher diversity.

Generate the improved top-10 recommendations.

### 4. Evaluation

Compare baseline and improved recommendations using the same user profiles.

Evaluate:

- Average relevance.
- Topic diversity.
- Country diversity.
- Publisher diversity.
- Interest Bridging coverage.

For topic diversity, consider all topic labels rather than relying only on `primary_topic`.

### 5. Streamlit Website

Build four pages:

- My Feed
- Beyond Your Circle
- Compare Feeds
- Feed Insights

### 6. User Feedback

Allow users to express interest in articles and update their learned preferences.

The updated preferences will be used to refresh the recommendations.

This will be implemented using a simple feedback-based algorithm rather than training a separate machine learning model.



## Current Status

**Baseline implementation: Completed.**

The next step is to develop Interest Bridging and diversity-aware re-ranking, then integrate and evaluate the improved recommendation system.